# Step 6 — Active Learning Iteration 2

**PDF pages 13–16.** Pick the 100 unlabeled users that the Step-5 winning model is least sure about, then label them manually.

**Key PDF clarification (page 14):**
> *"Step 6.1: Choose **THE model** (singular) with the best performance from the previous step. Apply **it** to the full unlabeled dataset."*

So Step 6 uses ONE model — not all three. The winner for the primary task `target_population` is **LogReg + numeric + K-Fold + unbalanced** (F1=0.665, AUC=0.747). This model uses **only numeric features** — no TF-IDF on translated text — so we don't need any translation at all.

**Pipeline:**
1. Load labeled (100) + filter the unlabeled pool from `Candidates_user_data_MERGED.csv` (846 users).
2. Compute the same 11 numeric features for both sets. The three Iran-keyword features use a **multilingual keyword list** (English + Persian + Arabic) on raw description / display_name / location, so we don't need to translate.
3. Re-train the LogReg + numeric `target_population` model on all 100 labeled examples (with the multilingual-keyword definition, so the feature is consistent between train and predict).
4. Predict on 846 unlabeled, compute confidence + uncertainty.
5. Save to `Iteration_2/`:
   - `iteration_2_unlabeled_users_predictions.csv` — every unlabeled user + the 6 PDF-mandated columns (`predicted_class`, `confidence_level`, `prob_0/1/2`, `uncertainty_score`).
   - `iteration_2_users_to_label.csv` — top 100 most uncertain, ready for manual labeling.

Total runtime: a few seconds.

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# --- Paths ---
folder    = Path.cwd()                                 # Classification/
DATA_DIR  = folder.parent / 'data'
ITER1_DIR = folder / 'Iteration_1' / 'Step5_Analysis'
ITER2_DIR = folder / 'Iteration_2'
ITER2_DIR.mkdir(exist_ok=True)

# --- Labeled pool (Step 3 output) ---
labeled = pd.read_csv(ITER1_DIR / 'iteration_1_consensus_translated.csv')
print(f"Labeled users: {len(labeled)}")

# --- Full candidate pool (the source the 100 labeled were drawn from) ---
pool = pd.read_csv(DATA_DIR / 'Candidates_user_data_MERGED.csv')
print(f"Full candidate pool: {len(pool)} users")

# --- Unlabeled = pool minus labeled, joined by username (case-insensitive) ---
labeled_keys = set(labeled['username'].astype(str).str.lower().str.strip())
unlabeled = pool[~pool['username'].astype(str).str.lower().str.strip().isin(labeled_keys)].copy()
unlabeled = unlabeled.reset_index(drop=True)
print(f"\nUnlabeled users (pool − labeled): {len(unlabeled)}")

Labeled users: 100
Full candidate pool: 946 users

Unlabeled users (pool − labeled): 846


In [17]:
# --- Compute the 11 numeric features WITHOUT translation. ---
# Iran keywords are matched in three languages so we catch the same users a translated
# bio would have flagged. We re-compute the feature on the LABELED set the same way
# so the train/predict feature definitions match.

iran_keywords_multilingual = [
    # English
    'iran', 'iranian', 'persian', 'persia', 'tehran', 'shiraz', 'esfahan', 'isfahan',
    'mashhad', 'tabriz', 'kerman', 'qom', 'farsi',
    # Persian / Farsi
    'ایران', 'ایرانی', 'تهران', 'شیراز', 'اصفهان', 'مشهد', 'تبریز', 'کرمان', 'قم', 'فارسی',
    # Arabic
    'إيران', 'ايران', 'إيراني', 'ايراني', 'طهران', 'شيراز',
]

def has_iran_multilingual(text):
    if pd.isna(text):
        return 0
    s = str(text).lower()
    return int(any(kw in s for kw in iran_keywords_multilingual))

def build_numeric_no_translation(df):
    """Build the same 11 numeric features Step 5 used, but without any translated columns."""
    df = df.copy()
    for c in ['followers_count', 'following_count', 'statuses_count']:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
    df['bio_length'] = df['description'].fillna('').astype(str).str.len()
    df['followers_following_ratio'] = df['followers_count'] / (df['following_count'] + 1)
    df['created_at_dt'] = pd.to_datetime(df['created_at'], format='%B %Y', errors='coerce')
    df['account_age_years'] = ((pd.Timestamp('2026-05-15') - df['created_at_dt']).dt.days / 365.25)
    df['has_description'] = df['description'].notna().astype(int)
    df['has_location']    = df['location'].notna().astype(int)
    df['bio_mentions_iran']      = df['description'].apply(has_iran_multilingual)
    df['name_mentions_iran']     = df['display_name'].apply(has_iran_multilingual)
    df['location_mentions_iran'] = df['location'].apply(has_iran_multilingual)
    return df

labeled   = build_numeric_no_translation(labeled)
unlabeled = build_numeric_no_translation(unlabeled)

# Fill any missing account_age_years with the labeled-set median
median_age = labeled['account_age_years'].median()
labeled['account_age_years']   = labeled['account_age_years'].fillna(median_age)
unlabeled['account_age_years'] = unlabeled['account_age_years'].fillna(median_age)

numeric_features = [
    'followers_count', 'following_count', 'statuses_count',
    'followers_following_ratio', 'bio_length', 'account_age_years',
    'has_description', 'has_location',
    'bio_mentions_iran', 'name_mentions_iran', 'location_mentions_iran',
]

print(f"Numeric features: {len(numeric_features)}")
print(f"  labeled[numeric]:   {labeled[numeric_features].shape}")
print(f"  unlabeled[numeric]: {unlabeled[numeric_features].shape}")
print(f"\nIran keyword hit rates (sanity check):")
for col in ['bio_mentions_iran', 'name_mentions_iran', 'location_mentions_iran']:
    print(f"  {col}: labeled={labeled[col].sum()}/{len(labeled)}  unlabeled={unlabeled[col].sum()}/{len(unlabeled)}")

Numeric features: 11
  labeled[numeric]:   (100, 11)
  unlabeled[numeric]: (846, 11)

Iran keyword hit rates (sanity check):
  bio_mentions_iran: labeled=7/100  unlabeled=60/846
  name_mentions_iran: labeled=1/100  unlabeled=9/846
  location_mentions_iran: labeled=10/100  unlabeled=76/846


In [18]:
# --- Train the Step 5 winner: LogReg + numeric, on all 100 labeled examples (3-class target_population) ---
import scipy.sparse as sp
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler(with_mean=False)
X_labeled   = sp.csr_matrix(scaler.fit_transform(labeled[numeric_features].values))
X_unlabeled = sp.csr_matrix(scaler.transform(unlabeled[numeric_features].values))

y_labeled = labeled['target_population'].values

model = LogisticRegression(max_iter=2000, class_weight=None, random_state=42)
model.fit(X_labeled, y_labeled)

print(f"Trained LogReg + numeric on {len(y_labeled)} labeled rows")
print(f"  Classes the model can predict: {list(model.classes_)}")
print(f"  Sanity-check training accuracy: {model.score(X_labeled, y_labeled):.3f}")

Trained LogReg + numeric on 100 labeled rows
  Classes the model can predict: [np.int64(0), np.int64(1), np.int64(2)]
  Sanity-check training accuracy: 0.730


In [19]:
# --- Predict on unlabeled, compute the PDF-mandated 6 columns (page 14) ---
proba = model.predict_proba(X_unlabeled)

# Map model.classes_ to fixed columns prob_0 / prob_1 / prob_2
proba_full = np.zeros((proba.shape[0], 3))
for j, c in enumerate(model.classes_):
    proba_full[:, int(c)] = proba[:, j]

predicted_class   = model.predict(X_unlabeled)
confidence_level  = proba_full.max(axis=1)
uncertainty_score = 1.0 - confidence_level

predictions = unlabeled[[
    'username', 'display_name', 'description',
    'location', 'followers_count', 'following_count', 'statuses_count', 'created_at',
]].copy()
predictions['predicted_class']    = predicted_class
predictions['confidence_level']   = confidence_level.round(4)
predictions['prob_0']             = proba_full[:, 0].round(4)
predictions['prob_1']             = proba_full[:, 1].round(4)
predictions['prob_2']             = proba_full[:, 2].round(4)
predictions['uncertainty_score']  = uncertainty_score.round(4)

# Sort by uncertainty descending (PDF: 'sort by uncertainty_score from highest to lowest')
predictions = predictions.sort_values('uncertainty_score', ascending=False).reset_index(drop=True)

print(f"Predictions for {len(predictions)} unlabeled users.\n")
print(f"uncertainty_score stats: min={predictions['uncertainty_score'].min():.3f}  "
      f"median={predictions['uncertainty_score'].median():.3f}  "
      f"max={predictions['uncertainty_score'].max():.3f}\n")
print(f"Predicted class distribution: "
      f"{dict(pd.Series(predicted_class).value_counts().sort_index())}")
print("\nTop 5 most uncertain:")
print(predictions[['username', 'description', 'location',
                    'predicted_class', 'confidence_level', 'uncertainty_score']].head(5).to_string(index=False))

Predictions for 846 unlabeled users.

uncertainty_score stats: min=0.000  median=0.244  max=0.645

Predicted class distribution: {0: np.int64(411), 1: np.int64(77), 2: np.int64(358)}

Top 5 most uncertain:
       username                                                                                                                                      description location  predicted_class  confidence_level  uncertainty_score
      afshin_46                                    نام من سید محمد حاجی سید جوادی است و مسئولیت حقوقی  نوشته هایم بر عهده من است.  #جمهوری‌خواه  #سوسیال‌دموکرات      NaN                2            0.3547             0.6453
alexanderm18395 He/him. Twin  Woke is an informal adjective that means being aware of and attentive to important issues, especially social and racial injustice.      NaN                1            0.3642             0.6358
abdelra07745930                                                  منشق من منظمة بدر وصاحب قناة على اليوتيوب ( قناة الأسير ا

In [20]:
# --- Save outputs ---
predictions_path = ITER2_DIR / 'iteration_2_unlabeled_users_predictions.csv'
predictions.to_csv(predictions_path, index=False)
print(f"Saved {len(predictions)} rows to: {predictions_path.relative_to(folder)}")

# Top 100 most uncertain — these are the users you and your partner will label manually
to_label = predictions.head(100).copy()

# Append empty label columns for the annotator to fill in (PDF page 15: all 3 target columns)
to_label['target_population']      = ''   # 0=non_target, 1=target, 2=unknown
to_label['locals_vs_diaspora']     = ''   # 0=diaspora,   1=local,  2=unknown
to_label['person_vs_organization'] = ''   # 0=organization, 1=person, 2=unknown
to_label['comments']               = ''

to_label_path = ITER2_DIR / 'iteration_2_users_to_label.csv'
to_label.to_csv(to_label_path, index=False)
print(f"Saved top-100 to-label file to: {to_label_path.relative_to(folder)}")

print("\nNext steps for you + your partner:")
print("  1. Open iteration_2_users_to_label.csv in Excel/Sheets")
print("  2. For each row, open https://x.com/<username> and inspect bio + tweets")
print("  3. Fill all 3 label columns (target_population, locals_vs_diaspora, person_vs_organization)")
print("  4. Both annotators label the same 100 users (Double Annotation), then we'll do consensus.")
print("  5. Save as iteration_2_manual_labels_*.csv (3 files like Step 3).")

Saved 846 rows to: Iteration_2/iteration_2_unlabeled_users_predictions.csv
Saved top-100 to-label file to: Iteration_2/iteration_2_users_to_label.csv

Next steps for you + your partner:
  1. Open iteration_2_users_to_label.csv in Excel/Sheets
  2. For each row, open https://x.com/<username> and inspect bio + tweets
  3. Fill all 3 label columns (target_population, locals_vs_diaspora, person_vs_organization)
  4. Both annotators label the same 100 users (Double Annotation), then we'll do consensus.
  5. Save as iteration_2_manual_labels_*.csv (3 files like Step 3).
